In [1]:
import pandas as pd

In [2]:
#df = pd.read_csv('gold_ml_price_features.csv')
df = pd.read_csv('alza_dataset.csv')

In [3]:
df.head(5)

,product_id,price,name,breadcrumb_section,breadcrumb_category,breadcrumb_subcategory,producer_id,eshop,sales,warranty,spec_summary,rating,in_stock,review_count,avg_rating,recommendation_rate,complaint_rate
0,3809599,8499.00,La Piccola Gold,Household and Personal Appliances,Coffee Makers and Presses,Lever,2845.0,Alza,24,24 months,"Lever Coffee Machine - pressure 18bar, water ...",4.9,t,5.0,4.9,1.00,NaN
1,3837335,549.00,KEMO repellent for mice and rodents,Car & Motorcycle,Car Accessories,Game Scarers and Repellers,15917.0,AutoMoto,3495,24 months,"- High frequency, 12V",4.3,t,29.0,4.3,0.80,0.0036
2,3753577,589.54,Philips Alcyone 77113/31/16,"House, Hobby and Garden",Light Bulbs and Lighting,NaN,1611.0,Alza,16,60 months,"Spot Lighting - LED, ceiling light, suspended ...",0.0,t,0.0,NaN,NaN,NaN
3,3809147,629.00,Epson T7605 Light Cyan,Computers and Laptops,Printers and Scanners,Toners & Inks,1301.0,Alza,14,24 months,"Cartridge - for SureColor SC-P600, 2400 pages",5.0,t,0.0,5.0,1.00,NaN
4,3837346,109.00,Spray against martens 200ml,Car & Motorcycle,Car Accessories,Game Scarers and Repellers,8610.0,AutoMoto,3385,24 months,"Marten Repellents Marten repellent, 200ml",4.5,t,19.0,4.5,0.84,0.0003


In [26]:
def create_textual_representation(row):
    subcategory = f"\n\nSubcategory: {row['breadcrumb_subcategory']}" if pd.notna(row["breadcrumb_subcategory"]) else ""
    textual_representation = f"""Product name: {row["name"]},

Price: {row["price"]},

Description: {row["spec_summary"]}

Category: {row["breadcrumb_category"]}{subcategory}


"""
    return textual_representation

In [27]:
df['textual_representation'] = df.apply(create_textual_representation, axis=1)

In [28]:
for i in range(10):
    print(df['textual_representation'].values[i])

Product name: La Piccola Gold,

Price: 8499.0,

Description: Lever Coffee Machine - pressure 18bar,  water container volume 1l,  input power 500W,  number of coffee beverages:,  width 15cm,  height 25cm,  depth 22cm,  weight 5kg,  colour: gold 

Category: Coffee Makers and Presses

Subcategory: Lever



Product name: KEMO repellent for mice and rodents,

Price: 549.0,

Description: - High frequency, 12V

Category: Car Accessories

Subcategory: Game Scarers and Repellers



Product name: Philips Alcyone 77113/31/16,

Price: 589.54,

Description: Spot Lighting - LED, ceiling light, suspended ceiling light, 230 V, luminous flux 630 lm, IP20 protection 

Category: Light Bulbs and Lighting



Product name: Epson T7605 Light Cyan,

Price: 629.0,

Description: Cartridge - for SureColor SC-P600, 2400 pages

Category: Printers and Scanners

Subcategory: Toners & Inks



Product name: Spray against martens 200ml,

Price: 109.0,

Description: Marten Repellents Marten repellent, 200ml

Category: C

In [29]:
import faiss
import requests
import numpy as np
import httpx
import asyncio
from tqdm.asyncio import tqdm

dim = 768 #emb dim - nomic-embed-text compatible

index = faiss.IndexFlatL2(dim)

X = np.zeros((len(df['textual_representation']), dim), dtype = 'float32')

In [30]:
async def get_embedding(client, text):
    res = await client.post('http://localhost:11434/api/embeddings',
                            json={'model': 'nomic-embed-text',
                                  'prompt': text
                                 }
                           )
    data = res.json()
    if 'embedding' not in data:
        print(f"Bad response: {data}, text: {text[:100]}")
        return [0.0] * 768  # return zeros as fallback
    return data['embedding']

async def build_embeddings(texts, concurrency=100):
    sem = asyncio.Semaphore(concurrency)
    async def get_with_sem(client, text):
        async with sem:
            return await get_embedding(client, text)
    
    async with httpx.AsyncClient(timeout=120) as client:
        tasks = [get_with_sem(client, t) for t in texts]
        return await tqdm.gather(*tasks, desc="Embedding")

embeddings = await build_embeddings(df['textual_representation'].tolist())
X = np.array(embeddings, dtype='float32')
index.add(X)

Embedding: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29708/29708 [05:41<00:00, 87.07it/s]


In [31]:
X

array([[-4.3218762e-01,  1.5487804e+00, -3.5243428e+00, ...,
        -8.0121532e-02, -6.6956651e-01, -3.9109948e-01],
       [ 5.3870583e-01, -1.6817005e-02, -3.9262578e+00, ...,
        -9.7417140e-01, -7.4266094e-01, -1.2144510e+00],
       [-1.5413325e-01,  2.3250982e-01, -3.4967306e+00, ...,
        -6.5592057e-01, -7.3150480e-01, -1.1420906e+00],
       ...,
       [-8.5574359e-02,  3.9555120e-01, -3.9197106e+00, ...,
         1.0861258e-03, -1.3039430e+00, -2.7503592e-01],
       [ 3.3304811e-01,  7.1340406e-01, -4.0112705e+00, ...,
        -2.9532081e-01, -9.8817015e-01, -4.1311964e-01],
       [-1.2507705e-01,  8.4144062e-01, -3.2292013e+00, ...,
         2.8160948e-01, -1.3334302e+00, -9.5745301e-01]],
      shape=(29708, 768), dtype=float32)

In [32]:
faiss.write_index(index, 'index')

### `index` is now a vector database full of embeddings
Lets load it

In [34]:
index = faiss.read_index('index')

### Bad response issua
TODO: investigate, maybe add truncation of text length

In [35]:
!ls

alza_dataset.csv          index
embeddings_pipeline.ipynb test_the_embeddings.ipynb
